# Diabetic retinopathy staging with RETFound (auto-detects MAE vs DINOv2)

**Update:** the "size mismatch" / "missing keys" error you hit means the
checkpoint you downloaded is actually the **RETFound-MAE** architecture
(patch size 16, has a `fc_norm` layer), not RETFound-DINOv2 (patch size 14,
uses LayerScale `ls1`/`ls2`). The benchmark folders bundle both variants, and
it's easy to grab the wrong one - so this version stops guessing and just
**detects which architecture your checkpoint actually is** from its weight
shapes, then builds that one. Same `score_image(path)` workflow either way.

It also now downloads the authors' own `models_vit.py` directly from the repo
and imports it, rather than me re-implementing the model class by hand - that
file defines both architectures exactly as they're used in training/eval, so
there's no risk of a subtle mismatch on my end.

Re-run all the setup cells once, then use the last cell as before.

**One-time install:** `pip install torch torchvision timm pillow requests`

In [9]:
from pathlib import Path

import torch
import torch.nn.functional as F
import timm
from PIL import Image
from torchvision import transforms

In [10]:
# Download the authors' own model-building code (models_vit.py) so the exact
# same RETFound_mae / RETFound_dinov2 architectures they use are used here too.
MODELS_VIT_URL = "https://raw.githubusercontent.com/rmaphoh/RETFound/main/models_vit.py"
MODELS_VIT_PATH = Path("models_vit.py")

if not MODELS_VIT_PATH.exists():
    import requests
    print("Downloading models_vit.py from the official repo...")
    r = requests.get(MODELS_VIT_URL, timeout=60)
    r.raise_for_status()
    MODELS_VIT_PATH.write_text(r.text, encoding="utf-8")
    print("Done.")

import models_vit

# Newer timm versions changed VisionTransformer.forward()'s internal call to
# forward_features(x, attn_mask=..., is_causal=...), but this repo's custom
# VisionTransformer class (written against timm~=0.9.2) only accepts (self, x).
# Patch it to silently ignore any extra args/kwargs newer timm passes in -
# the underlying computation is untouched, this only fixes the call signature.
_orig_forward_features = models_vit.VisionTransformer.forward_features
def _compat_forward_features(self, x, *args, **kwargs):
    return _orig_forward_features(self, x)
models_vit.VisionTransformer.forward_features = _compat_forward_features

# models_vit.VisionTransformer.forward_features() (RETFound_mae) already does
# its own pooling internally: global_pool=True -> mean-pool + fc_norm, giving
# an already-pooled [B, 1, D] (or [B, D]) tensor; global_pool=False -> norm +
# cls-token, giving [B, D]. The old timm this was written against then just
# did head(forward_features(x)). Modern timm's forward() instead calls
# forward_head() -> pool(), which tries to use self.global_pool (a bool here,
# e.g. True) as a pool-type *string* like 'avg'/'token' and raises
# `AssertionError: Unknown pool type True`. Bypass that re-pooling entirely
# and go straight to the head, as the original code intends.
def _compat_forward(self, x, *args, **kwargs):
    x = self.forward_features(x)
    if x.dim() == 3 and x.shape[1] == 1:
        x = x.squeeze(1)
    x = self.head_drop(x)
    return self.head(x)
models_vit.VisionTransformer.forward = _compat_forward


In [11]:
# ==== EDIT THIS: path to the checkpoint you downloaded ====
CKPT_PATH = r"C:\Users\adity\Desktop\manit_assignmentes\mldl\mldl_lab\assignment_3\xai-dr-screening\retfound_mae\checkpoint-best.pth"

NUM_CLASSES = 5
IMG_SIZE = 224

# Standard ICDR diabetic retinopathy grading scale used by APTOS2019 / IDRiD / MESSIDOR2.
DR_LABELS = [
    "0 - No DR",
    "1 - Mild NPDR",
    "2 - Moderate NPDR",
    "3 - Severe NPDR",
    "4 - Proliferative DR",
]

In [12]:
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _detect_variant(state_dict):
    """
    Tell RETFound-MAE and RETFound-DINOv2 checkpoints apart from their weight
    shapes alone, so we build whichever architecture actually matches:
      - MAE:    patch_embed.proj.weight is [.., 16, 16], has fc_norm, no LayerScale
      - DINOv2: patch_embed.proj.weight is [.., 14, 14], has blocks.*.ls1/ls2.gamma
    """
    w = state_dict.get("patch_embed.proj.weight")
    if w is None:
        raise ValueError("Could not find patch_embed.proj.weight in the checkpoint - unexpected format.")
    patch_size = w.shape[-1]
    if patch_size == 16:
        return "mae"
    elif patch_size == 14:
        return "dinov2"
    raise ValueError(
        f"Unrecognised patch size {patch_size} in checkpoint - doesn't look like a "
        f"RETFound-MAE or RETFound-DINOv2 checkpoint."
    )

In [13]:
def build_and_load_model(ckpt_path, num_classes: int = NUM_CLASSES):
    ckpt_path = Path(ckpt_path)
    if not ckpt_path.is_file():
        raise FileNotFoundError(
            f"Checkpoint not found: {ckpt_path}\n"
            f"Download a DINOv2 or MAE checkpoint for your chosen dataset and point CKPT_PATH at it."
        )

    # weights_only=False: PyTorch >=2.6 defaults to weights_only=True, which refuses
    # to unpickle the argparse.Namespace object saved alongside the weights in this
    # checkpoint. Safe here since this is the official benchmark checkpoint you
    # downloaded directly from the authors' Google Drive link.
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = checkpoint["model"] if isinstance(checkpoint, dict) and "model" in checkpoint else checkpoint

    variant = _detect_variant(state_dict)
    print(f"Detected checkpoint architecture: RETFound-{variant.upper()}")

    if variant == "mae":
        model = models_vit.RETFound_mae(
            img_size=IMG_SIZE, num_classes=num_classes, drop_path_rate=0.2, global_pool=True,
        )
    else:
        # RETFound_dinov2(args, **kwargs) doesn't actually use args - safe to pass None.
        # Note: this pulls Meta's DINOv2 ImageNet weights via timm on first call (one-time
        # download, ~1.2GB) before we immediately overwrite them with the checkpoint below.
        model = models_vit.RETFound_dinov2(None, num_classes=num_classes, drop_path_rate=0.2)

    try:
        model.load_state_dict(state_dict, strict=True)
    except RuntimeError as e:
        raise RuntimeError(
            f"Checkpoint looked like RETFound-{variant.upper()} (from its patch size) but still "
            f"didn't load cleanly. Double check NUM_CLASSES above matches how this checkpoint "
            f"was fine-tuned (should be 5 for DR grading).\nOriginal error: {e}"
        )
    return model

In [14]:
# Eval-time preprocessing, copied exactly from util/datasets.py build_transform():
#   crop_pct = 224/256 for input_size <= 224  ->  resize to 256, then center-crop to 224
_eval_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # IMAGENET_DEFAULT_MEAN/STD
])

# Cached across calls so score_image() is cheap to call repeatedly.
_MODEL = None
_DEVICE = None


def _get_model():
    global _MODEL, _DEVICE
    if _MODEL is None:
        _DEVICE = get_device()
        print(f"Loading checkpoint on {_DEVICE} (first call only)...")
        model = build_and_load_model(CKPT_PATH)
        model.eval().to(_DEVICE)
        _MODEL = model
    return _MODEL, _DEVICE

In [15]:
def score_image(image_path) -> dict:
    """
    Run RETFound DR-grading on a single fundus image.
    Returns per-stage probabilities plus the predicted stage.
    """
    image_path = Path(image_path)
    if not image_path.is_file():
        raise FileNotFoundError(
            f"Image not found: {image_path}\n"
            f"(If this is a Windows path, use a raw string, e.g. r'{image_path}', or forward slashes.)"
        )

    model, device = _get_model()
    img = Image.open(image_path).convert("RGB")
    tensor = _eval_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(tensor)
        probs = F.softmax(logits, dim=1).squeeze(0).cpu().tolist()

    predicted_idx = int(max(range(NUM_CLASSES), key=lambda i: probs[i]))
    return {
        "image": str(image_path),
        "predicted_stage": DR_LABELS[predicted_idx],
        "probabilities": {DR_LABELS[i]: round(probs[i], 4) for i in range(NUM_CLASSES)},
    }

## Check an image

Edit `image_path` below (use a raw string `r"..."` or forward slashes for
Windows paths), then re-run this cell for each image you want to check.
The model loads once on the first run and stays cached, so later checks
are fast.

In [ ]:
image_path = r"C:\Users\adity\Desktop\manit_assignmentes\mldl\mldl_lab\assignment_3\xai-dr-screening\dataset_classificiation\train_images\train_images\1b8ad0afe9fb.png"

result = score_image(image_path)
print("Predicted stage:", result["predicted_stage"])
print("\nFull probability breakdown:")
for label, p in result["probabilities"].items():
    print(f"  {label}: {p:.1%}")

Loading checkpoint on cuda (first call only)...
Detected checkpoint architecture: RETFound-MAE
